# Dynamic Bernoulli Embeddings for Sinhala
This notebook implements Dynamic Bernoulli Embeddings within a standalone environment, dropping dependencies to the original repository. It processes your `century_lemmas.json` directly.


In [1]:
pip install scikit-learn tensorflow

  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached keras-3.14.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import glob
import json
import numpy as np
import pandas as pd
import pickle
import time
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
from tensorflow.compat.v1.distributions import Normal, Bernoulli

# Seed for reproducibility
np.random.seed(42)
tf.set_random_seed(42)



Instructions for updating:
non-resource variables are not supported in the long term


In [3]:
def plot_with_labels(low_dim_embs, labels, fname):
    plt.figure(figsize=(28, 28))
    for i, label in enumerate(labels):
        x, y = low_dim_embs[i, :]
        plt.scatter(x, y)
        plt.annotate(label, xy=(x, y), xytext=(5, 2),
                     textcoords='offset points', ha='right', va='bottom',
                     fontproperties='Sinhala Sangam MN', fontsize=12)
    plt.savefig(fname)
    plt.close()

def make_dir(name):
    if not os.path.exists('fits'): os.makedirs('fits')
    dir_name = os.path.join('fits', name, 'EF_EMB_'+time.strftime("%y_%m_%d_%H_%M_%S"))
    while os.path.isdir(dir_name):
        time.sleep(np.random.randint(10))
        dir_name = os.path.join('fits', name, 'EF_EMB_'+time.strftime("%y_%m_%d_%H_%M_%S"))
    os.makedirs(dir_name)
    return dir_name

def variable_summaries(summary_name, var):
    with tf.name_scope(summary_name):
        mean = tf.reduce_mean(var)
        tf.summary.scalar('mean', mean)
        with tf.name_scope('stddev'):
            stddev = tf.sqrt(tf.reduce_mean(tf.square(var - mean)))
        tf.summary.scalar('stddev', stddev)
        tf.summary.scalar('max', tf.reduce_max(var))
        tf.summary.scalar('min', tf.reduce_min(var))



In [4]:
def preprocess_dataset(json_path, dataset_name='dbe_sinhala', V=5000):
    # 1. Read JSON
    print("Parsing centuries and counts...")
    with open(json_path, 'r', encoding='utf-8') as f:
        century_lemmas = json.load(f)
        
    # Create directories
    base_dir = os.path.join('dat', dataset_name)
    for sub in ['raw', 'train', 'valid', 'test']:
        os.makedirs(os.path.join(base_dir, sub), exist_ok=True)
        
    allowed_centuries = [13, 14, 15, 18, 19, 20]
    time_slices = sorted([int(k) for k in century_lemmas.keys() if k.isdigit() and int(k) in allowed_centuries])
    print(f"Selected Centuries: {time_slices}")
    
    # Z-Score Stopword Filtering
    print("Calculating Z-scores to identify stopwords...")
    freq_by_century = {str(c): {} for c in time_slices}
    for century in time_slices:
        for sentence in century_lemmas[str(century)]:
            for word in sentence:
                freq_by_century[str(century)][word] = freq_by_century[str(century)].get(word, 0) + 1
                
    freq_df = pd.DataFrame(freq_by_century).fillna(0).astype(int)
    zscore_df = (freq_df - freq_df.mean()) / freq_df.std(ddof=0)
    
    stop_words = set()
    for century in zscore_df.columns:
        outliers = zscore_df[zscore_df[century] > 6.1027].index.tolist()
        stop_words.update(outliers)
        
    identified_non_stopwords = ['ධර්ම', 'ප්රකාශ', 'පිණිස', 'බුද්ධ', 'බුදු', 'බුදුන්', 'බොහෝ', 'මහ', 'මහත්', 'රජ', 'යුක්ත', 'වදාළ', 'වහන්සේ', 'වැඩ', 'ශ්රී', 'සතර', 'සඳ', 'සමග', 'සිත', 'සිත්', 'සියලු']
    stop_words.difference_update(identified_non_stopwords)
        
    print(f"Identified {len(stop_words)} stopwords across centuries using Z-score > 6.1027. Filtering...")
    
    # Filter stopwords from sentences (squashing context windows)
    for century in time_slices:
        filtered_text_data = []
        for sentence in century_lemmas[str(century)]:
            filtered_sentence = [w for w in sentence if w not in stop_words]
            if filtered_sentence:
                filtered_text_data.append(filtered_sentence)
        century_lemmas[str(century)] = filtered_text_data
    
    prefix_length = 2
    
    # Word counts
    words_list = []
    for century in time_slices:
        text_data = century_lemmas[str(century)]
        for sentence in text_data:
            words_list.extend(sentence)
    
    dictionary = {}
    count = {}
    print("Building Dictionary...")
    for word in words_list:
        if word not in dictionary:
            count[len(dictionary)] = 1
            dictionary[word] = len(dictionary)
        else:
            count[dictionary[word]] += 1
            
    # Truncate Vocabulary
    df = pd.DataFrame.from_dict(dictionary, orient='index', columns=['idx'])
    cf = pd.DataFrame.from_dict(count, orient='index', columns=['cnt'])
    uni = df.join(cf, on='idx').sort_values(by='cnt', ascending=False).reset_index()
    uni.columns = ['word', 'old_idx', 'cnt']
    uni = uni.reset_index()
    uni.columns = ['new_idx', 'word', 'old_idx', 'cnt']
    
    old_idx = uni.old_idx.values
    V = min(V, len(old_idx))
    print(f"Vocab size truncated to: {V}")
    
    uni.head(V).to_csv(os.path.join(base_dir, 'unigram.txt'), header=False, index=False, sep='\t', columns=['word', 'new_idx', 'cnt'])
    
    # Precompute mapping (old_idx -> new_idx)
    mapping = np.zeros(len(old_idx), dtype=np.int32) + 2*V
    for ni, oi in enumerate(old_idx[:V]):
        mapping[oi] = ni
    
    # Save raw numpy files and splits
    dat_stats = {
        'name': dataset_name,
        'T_bins': time_slices,
        'prefix': prefix_length,
        'query_words': [], # Add custom queries if needed
        'train': np.zeros(len(time_slices)),
        'test': np.zeros(len(time_slices)),
        'valid': np.zeros(len(time_slices))
    }
    
    for t, century in enumerate(time_slices):
        word_indices = []
        text_data = century_lemmas[str(century)]
        for sentence in text_data:
            for word in sentence:
                val = mapping[dictionary[word]]
                if val < V:
                    word_indices.append(val)
        
        data = np.array(word_indices, dtype=np.int32)
        n_data = len(data)
        
        # Split: 80% train, 10% valid, 10% test
        n_valid = int(n_data * 0.1)
        n_test = int(n_data * 0.1)
        n_train = n_data - n_valid - n_test
        
        dat_stats['train'][t] = n_train
        dat_stats['valid'][t] = n_valid
        dat_stats['test'][t] = n_test
        
        # Save files
        prefix_str = f"{century:02d}"
        np.save(os.path.join(base_dir, 'train', f"{prefix_str}_dat.npy"), data[:n_train])
        np.save(os.path.join(base_dir, 'valid', f"{prefix_str}_dat.npy"), data[n_train:n_train+n_valid])
        np.save(os.path.join(base_dir, 'test', f"{prefix_str}_dat.npy"), data[n_train+n_valid:])
        
    with open(os.path.join(base_dir, 'dat_stats.pkl'), 'wb') as f:
        pickle.dump(dat_stats, f)
        
    print("Data processing complete.")



In [5]:
class bern_emb_data:
    def __init__(self, cs, ns, fpath, dynamic, n_epochs=1):
        assert cs % 2 == 0
        self.cs = cs
        self.ns = ns
        self.n_epochs = n_epochs
        self.dynamic = dynamic
        with open(os.path.join(fpath, "dat_stats.pkl"), "rb") as f:
            dat_stats = pickle.load(f)
        self.T = len(dat_stats['T_bins'])
        self.name = dat_stats['name']
        
        if not self.dynamic:
            self.N = int(np.sum(dat_stats['train']))
            self.n_train = int(self.N / n_epochs)
            self.n_valid = int(np.sum(dat_stats['valid']))
            self.n_test = int(np.sum(dat_stats['test']))
        else:
            self.N = int(np.sum(dat_stats['train']))
            self.n_train = np.maximum(dat_stats['train'] / n_epochs, 2 * np.ones_like(dat_stats['train'])).astype(np.int32)
            self.n_valid = dat_stats['valid'].astype(np.int32)
            self.n_test = dat_stats['test'].astype(np.int32)

        # load vocabulary
        df = pd.read_csv(os.path.join(fpath, 'unigram.txt'), delimiter='\t', header=None)
        self.labels = df[0].values
        self.counts = df[df.shape[1]-1].values
        counts_scaled = (1.0 * self.counts / self.N) ** (3.0 / 4)
        self.unigram = counts_scaled / counts_scaled.sum() # Normalized
        
        self.L = len(self.labels)
        self.dictionary = dict(zip(self.labels, range(self.L)))
        self.query_words = [w for w in dat_stats['query_words'] if w in self.labels]

        def make_batches(split, n_counts):
            files = glob.glob(os.path.join(fpath, split, '*.npy'))
            if self.dynamic:
                batches = [0] * self.T
                for t, i in enumerate(dat_stats['T_bins']):
                    filtered = [f for f in files if int(os.path.basename(f)[:dat_stats['prefix']]) == i]
                    batches[t] = self.batch_generator(int(n_counts[t]) + self.cs, filtered)
                return batches
            else:
                return self.batch_generator(int(n_counts) + self.cs, files)

        self.batch = make_batches('train', self.n_train)
        self.valid_batch = make_batches('valid', self.n_valid)
        self.test_batch = make_batches('test', self.n_test)

    def batch_generator(self, batch_size, files):
        if not files:
            while True: yield np.zeros(batch_size, dtype=np.int32)
        f_idx = 0
        data = np.load(files[f_idx])
        while True:
            if data.shape[0] < batch_size:
                f_idx = (f_idx + 1) % len(files)
                data_new = np.load(files[f_idx])
                data = np.hstack([data, data_new])
                if data.shape[0] < batch_size:
                    continue
            words = data[:batch_size]
            data = data[batch_size:]
            yield words
    
    def train_feed(self, placeholder):
        if self.dynamic:
            return {placeholder[t]: next(self.batch[t]) for t in range(self.T)}
        return {placeholder: next(self.batch)}
        
    def valid_feed(self, placeholder):
        if self.dynamic:
            return {placeholder[t]: next(self.valid_batch[t]) for t in range(self.T)}
        return {placeholder: next(self.valid_batch)}



In [6]:
class Config:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

class emb_model(object):
    def __init__(self, args, d, logdir):
        self.args = args
        self.K = args.K
        self.cs = args.cs
        self.ns = args.ns
        self.sig = args.sig
        self.dynamic = args.dynamic
        self.logdir = logdir
        self.N = d.N
        self.L = d.L
        self.T = d.T
        self.n_minibatch = d.n_train
        self.n_test = d.n_test
        self.labels = d.labels
        self.unigram = d.unigram
        self.dictionary = d.dictionary
        self.query_words = d.query_words
        self.train_feed = d.train_feed
        self.n_iter = args.n_iter
        self.n_epochs = d.n_epochs
        self.alpha_trainable = True
        
        if args.init:
            fname = os.path.join('fits', d.name, args.init)
            if 'alpha_constant' in args.init:
                self.alpha_trainable = False
                fname = fname.replace('/alpha_constant','')
            with open(fname, 'rb') as f:
                fit = pickle.load(f)
            self.rho_init = fit['rho']
            self.alpha_init = fit['alpha']
        else:
            self.rho_init = (np.random.randn(self.L, self.K)/self.K).astype('float32')
            self.alpha_init = (np.random.randn(self.L, self.K)/self.K).astype('float32')
        
        if not self.alpha_trainable:
            self.rho_init = (0.1*np.random.randn(self.L, self.K)/self.K).astype('float32')

    def dump(self, fname): raise NotImplementedError()
    def detect_drift(self): raise NotImplementedError()

    def initialize_training(self):
        optimizer = tf.train.AdamOptimizer()
        self.train = optimizer.minimize(self.loss)
        self.sess = tf.Session()
        with self.sess.as_default():
            tf.global_variables_initializer().run()

        variable_summaries('alpha', self.alpha)
        with tf.name_scope('objective'):
            tf.summary.scalar('loss', self.loss)
            tf.summary.scalar('priors', self.log_prior)
            tf.summary.scalar('ll_pos', self.ll_pos)
            tf.summary.scalar('ll_neg', self.ll_neg)
        self.summaries = tf.summary.merge_all()
        self.train_writer = tf.summary.FileWriter(self.logdir, self.sess.graph)
        self.saver = tf.train.Saver()

    def train_embeddings(self):
        for data_pass in range(self.n_iter):
            for step in range(self.n_epochs):
                if step % max(1, self.n_epochs//10) == 0:
                    print(f"Step {step}/{self.n_epochs} (Iter {data_pass})")
                    summary, _ = self.sess.run([self.summaries, self.train], feed_dict=self.train_feed(self.placeholders))
                    self.train_writer.add_summary(summary, data_pass * self.n_epochs + step)
                else:
                    self.sess.run([self.train], feed_dict=self.train_feed(self.placeholders))
            
            self.dump(os.path.join(self.logdir, f"variational{data_pass}.dat"))
            self.saver.save(self.sess, os.path.join(self.logdir, "model.ckpt"), global_step=data_pass)

        self.plot_params(min(500, self.L))
        
class bern_emb_model(emb_model):
    def __init__(self, args, d, logdir):
        super().__init__(args, d, logdir)
        
        with tf.name_scope('model'):
            with tf.name_scope('input'):
                self.placeholders = tf.placeholder(tf.int32)
                self.words = self.placeholders

            with tf.name_scope('context_mask'):
                p_start = self.cs // 2
                p_end = self.n_minibatch + self.cs // 2
                self.p_mask = tf.cast(tf.range(p_start, p_end), tf.int32)
                rows = tf.cast(tf.tile(tf.expand_dims(tf.range(0, self.cs // 2), [0]), [self.n_minibatch, 1]), tf.int32)
                columns = tf.cast(tf.tile(tf.expand_dims(tf.range(0, self.n_minibatch), [1]), [1, self.cs // 2]), tf.int32)
                self.ctx_mask = tf.concat([rows + columns, rows + columns + self.cs // 2 + 1], 1)

            with tf.name_scope('embeddings'):
                self.rho = tf.Variable(self.rho_init, name='rho')
                self.alpha = tf.Variable(self.alpha_init, name='alpha', trainable=self.alpha_trainable)
                with tf.name_scope('priors'):
                    prior = Normal(loc=0.0, scale=self.sig)
                    if self.alpha_trainable:
                        self.log_prior = tf.reduce_sum(prior.log_prob(self.rho)) + tf.reduce_sum(prior.log_prob(self.alpha))
                    else:
                        self.log_prior = tf.reduce_sum(prior.log_prob(self.rho))

            with tf.name_scope('natural_param'):
                self.p_idx = tf.gather(self.words, self.p_mask)
                self.p_rho = tf.squeeze(tf.gather(self.rho, self.p_idx))

                unigram_logits = tf.tile(tf.expand_dims(tf.math.log(tf.constant(self.unigram, dtype=tf.float32)), [0]), [self.n_minibatch, 1])
                self.n_idx = tf.cast(tf.random.categorical(unigram_logits, self.ns), tf.int32)
                self.n_rho = tf.gather(self.rho, self.n_idx)

                self.ctx_idx = tf.squeeze(tf.gather(self.words, self.ctx_mask))
                self.ctx_alphas = tf.gather(self.alpha, self.ctx_idx)

                ctx_sum = tf.reduce_sum(self.ctx_alphas, [1])
                self.p_eta = tf.expand_dims(tf.reduce_sum(tf.multiply(self.p_rho, ctx_sum), -1), 1)
                self.n_eta = tf.reduce_sum(tf.multiply(self.n_rho, tf.tile(tf.expand_dims(ctx_sum, 1), [1, self.ns, 1])), -1)

            self.y_pos = Bernoulli(logits=self.p_eta)
            self.y_neg = Bernoulli(logits=self.n_eta)
            self.ll_pos = tf.reduce_sum(self.y_pos.log_prob(1.0))
            self.ll_neg = tf.reduce_sum(self.y_neg.log_prob(0.0))
            self.log_likelihood = self.ll_pos + self.ll_neg
            
            self.loss = - (self.n_epochs * self.log_likelihood + self.log_prior)

    def dump(self, fname):
        with self.sess.as_default():
            dat = {'rho': self.rho.eval(), 'alpha': self.alpha.eval()}
        with open(fname, "wb") as f:
            pickle.dump(dat, f)

    def plot_params(self, plot_only=500):
        with self.sess.as_default():
            tsne = TSNE(perplexity=30, n_components=2, init='pca', max_iter=2000)
            low_dim_embs_alpha = tsne.fit_transform(self.alpha.eval()[:plot_only])
            plot_with_labels(low_dim_embs_alpha, self.labels[:plot_only], os.path.join(self.logdir, 'alpha.png'))

            tsne = TSNE(perplexity=30, n_components=2, init='pca', max_iter=2000)
            low_dim_embs_rho = tsne.fit_transform(self.rho.eval()[:plot_only])
            plot_with_labels(low_dim_embs_rho, self.labels[:plot_only], os.path.join(self.logdir, 'rho.png'))

class dynamic_bern_emb_model(emb_model):
    def __init__(self, args, d, logdir):
        super().__init__(args, d, logdir)
        with tf.name_scope('model'):
            with tf.name_scope('embeddings'):
                self.alpha = tf.Variable(self.alpha_init, name='alpha', trainable=self.alpha_trainable)
                self.rho_t = {}
                for t in range(-1, self.T):
                    init_val = self.rho_init + 0.001 * np.random.randn(self.L, self.K).astype('float32') / self.K
                    if t == -1 and not self.alpha_trainable and args.init:
                        init_val = self.rho_init
                    self.rho_t[t] = tf.Variable(init_val, name=f'rho_{t}')

                with tf.name_scope('priors'):
                    global_prior = Normal(loc=0.0, scale=self.sig)
                    local_prior = Normal(loc=0.0, scale=self.sig / 100.0)
                    self.log_prior = tf.reduce_sum(global_prior.log_prob(self.alpha))
                    self.log_prior += tf.reduce_sum(global_prior.log_prob(self.rho_t[-1]))
                    for t in range(self.T):
                        self.log_prior += tf.reduce_sum(local_prior.log_prob(self.rho_t[t] - self.rho_t[t-1]))

            with tf.name_scope('likelihood'):
                self.placeholders = {}
                self.y_pos = {}
                self.y_neg = {}
                self.ll_pos = 0.0
                self.ll_neg = 0.0
                
                unigram_logits = tf.math.log(tf.constant(self.unigram, dtype=tf.float32))

                for t in range(self.T):
                    p_start = self.cs // 2
                    p_end = self.n_minibatch[t] + self.cs // 2
                    p_mask = tf.range(p_start, p_end)
                    rows = tf.tile(tf.expand_dims(tf.range(0, self.cs // 2), [0]), [self.n_minibatch[t], 1])
                    columns = tf.tile(tf.expand_dims(tf.range(0, self.n_minibatch[t]), [1]), [1, self.cs // 2])
                    ctx_mask = tf.concat([rows + columns, rows + columns + self.cs // 2 + 1], 1)

                    self.placeholders[t] = tf.placeholder(tf.int32, shape=(self.n_minibatch[t] + self.cs))
                    p_idx = tf.gather(self.placeholders[t], p_mask)
                    ctx_idx = tf.squeeze(tf.gather(self.placeholders[t], ctx_mask))

                    n_idx = tf.cast(tf.random.categorical(tf.tile(tf.expand_dims(unigram_logits, [0]), [self.n_minibatch[t], 1]), self.ns), tf.int32)
                    ctx_alphas = tf.gather(self.alpha, ctx_idx)

                    p_rho = tf.squeeze(tf.gather(self.rho_t[t], p_idx))
                    n_rho = tf.gather(self.rho_t[t], n_idx)

                    ctx_sum = tf.reduce_sum(ctx_alphas, [1])
                    p_eta = tf.expand_dims(tf.reduce_sum(tf.multiply(p_rho, ctx_sum), -1), 1)
                    n_eta = tf.reduce_sum(tf.multiply(n_rho, tf.tile(tf.expand_dims(ctx_sum, 1), [1, self.ns, 1])), -1)

                    self.y_pos[t] = Bernoulli(logits=p_eta)
                    self.y_neg[t] = Bernoulli(logits=n_eta)
                    self.ll_pos += tf.reduce_sum(self.y_pos[t].log_prob(1.0))
                    self.ll_neg += tf.reduce_sum(self.y_neg[t].log_prob(0.0))

            self.loss = - (self.n_epochs * (self.ll_pos + self.ll_neg) + self.log_prior)

    def dump(self, fname):
        with self.sess.as_default():
            dat = {'alpha': self.alpha.eval()}
            for t in range(self.T): dat[f'rho_{t}'] = self.rho_t[t].eval()
        with open(fname, "wb") as f:
            pickle.dump(dat, f)

    def plot_params(self, plot_only=500):
        with self.sess.as_default():
            tsne = TSNE(perplexity=30, n_components=2, init='pca', max_iter=2000)
            low_dim_embs_alpha = tsne.fit_transform(self.alpha.eval()[:plot_only])
            plot_with_labels(low_dim_embs_alpha, self.labels[:plot_only], os.path.join(self.logdir, 'alpha.png'))
            for t in [0, self.T // 2, self.T - 1]:
                np_rho = self.rho_t[t].eval()
                tsne = TSNE(perplexity=30, n_components=2, init='pca', max_iter=2000)
                low_dim_embs_rho = tsne.fit_transform(np_rho[:plot_only, :])
                plot_with_labels(low_dim_embs_rho, self.labels[:plot_only], os.path.join(self.logdir, f'rho_{t}.png'))



## 1. Extract and Process Dataset


In [8]:
# 1. Start Preprocessing
preprocess_dataset("../Data/century_lemmas.json", dataset_name="dbe_sinhala", V=5000)



Parsing centuries and counts...
Selected Centuries: [13, 14, 15, 18, 19, 20]
Calculating Z-scores to identify stopwords...
Identified 91 stopwords across centuries using Z-score > 6.1027. Filtering...
Building Dictionary...
Vocab size truncated to: 5000
Data processing complete.


## 2. Train Static Base Embeddings


In [9]:
# 2. Train Static Base Embeddings First
print("Training Static Embeddings...")
tf.reset_default_graph()

args_static = Config(
    K=100, cs=4, ns=10, n_epochs=10, 
    n_iter=1, sig=1.0, dynamic=False, init=None
)

d_static = bern_emb_data(args_static.cs, args_static.ns, "dat/dbe_sinhala", args_static.dynamic, args_static.n_epochs)
log_dir_static = make_dir(d_static.name)

m_static = bern_emb_model(args_static, d_static, log_dir_static)
m_static.initialize_training()

# Create dummy alpha_constant folder
alpha_dir = os.path.join(log_dir_static, 'alpha_constant')
if not os.path.exists(alpha_dir): os.makedirs(alpha_dir)

m_static.train_embeddings()

# Save to the specific 'alpha_constant/variational0.dat' path for the dynamic init
m_static.dump(os.path.join(alpha_dir, "variational0.dat"))
print("Static base model saved at:", alpha_dir)
static_alpha_rel_path = os.path.relpath(os.path.join(alpha_dir, "variational0.dat"), os.path.join('fits', d_static.name))



Training Static Embeddings...
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.


I0000 00:00:1777294221.722668 32960555 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Step 0/10 (Iter 0)
Step 1/10 (Iter 0)
Step 2/10 (Iter 0)
Step 3/10 (Iter 0)
Step 4/10 (Iter 0)
Step 5/10 (Iter 0)
Step 6/10 (Iter 0)
Step 7/10 (Iter 0)
Step 8/10 (Iter 0)
Step 9/10 (Iter 0)
Static base model saved at: fits/dbe_sinhala/EF_EMB_26_04_27_18_20_21/alpha_constant


## 3. Train Dynamic Temporal Embeddings


In [10]:
# 3. Train Dynamic Embeddings
print("Training Dynamic Embeddings initialized with static model alphas...")
tf.reset_default_graph()

args_dynamic = Config(
    K=100, cs=4, ns=10, n_epochs=10, 
    n_iter=1, sig=1.0, dynamic=True, init=static_alpha_rel_path
)

d_dynamic = bern_emb_data(args_dynamic.cs, args_dynamic.ns, "dat/dbe_sinhala", args_dynamic.dynamic, args_dynamic.n_epochs)
log_dir_dynamic = make_dir(d_dynamic.name + '_dynamic')

m_dynamic = dynamic_bern_emb_model(args_dynamic, d_dynamic, log_dir_dynamic)
m_dynamic.initialize_training()
m_dynamic.train_embeddings()

print("Dynamic Embeddings output stored in:", log_dir_dynamic)



Training Dynamic Embeddings initialized with static model alphas...
Step 0/10 (Iter 0)
Step 1/10 (Iter 0)
Step 2/10 (Iter 0)
Step 3/10 (Iter 0)
Step 4/10 (Iter 0)
Step 5/10 (Iter 0)
Step 6/10 (Iter 0)
Step 7/10 (Iter 0)
Step 8/10 (Iter 0)
Step 9/10 (Iter 0)
Dynamic Embeddings output stored in: fits/dbe_sinhala_dynamic/EF_EMB_26_04_27_18_20_28


## 4. Century-by-Century Evolution
Let's trace the nearest neighbors across centuries for a few selected words.

In [11]:
import os
import glob
import pickle
import numpy as np
import pandas as pd

# 1. Load vocabulary safely
df = pd.read_csv("dat/dbe_sinhala/unigram.txt", sep='\t', header=None)
labels = df[0].values
dictionary = {str(w): i for i, w in enumerate(labels)}

# 2. Dynamically find the latest trained weights from the fits directory
dynamic_dirs = glob.glob("fits/dbe_sinhala_dynamic/*")
if not dynamic_dirs:
    print("No trained embeddings found. Please run the training cells first.")
else:
    latest_dir = max(dynamic_dirs, key=os.path.getctime)
    dat_file = os.path.join(latest_dir, "variational0.dat")
    
    print(f"Loading data from {dat_file}...")
    with open(dat_file, "rb") as f:
        dat = pickle.load(f)
    with open("dat/dbe_sinhala/dat_stats.pkl", "rb") as fs:
        dat_stats = pickle.load(fs)
    actual_centuries = dat_stats['T_bins']

        
    alpha = dat['alpha']
    # L2 Normalize the global alpha context mapping
    norm_alpha = alpha / (np.linalg.norm(alpha, axis=1, keepdims=True) + 1e-10)
    T = len([k for k in dat.keys() if k.startswith('rho_')])
    
    query_words = ['බුදුන්', 'රජ', 'ධර්ම', 'මාර්ග', 'සිංහල']
    
    for x in query_words:
        if x not in dictionary:
            print(f"\nEvolution data for '{x}' not found (Not in vocab).")
            continue
            
        word_idx = dictionary[x]
        print(f"\n=====================================")
        print(f"Nearest Neighbors Evolution: {x}")
        print(f"=====================================")
        
        for t_idx in range(T):
            rho_t = dat[f'rho_{t_idx}']
            # L2 normalize period-specific embeddings (rho_t for word mapping)
            norm_rho_t = rho_t / (np.linalg.norm(rho_t, axis=1, keepdims=True) + 1e-10)
            
            # Compute similarity dot product
            query_vec = norm_rho_t[word_idx]
            # Compare target word against all OTHER target words at time t
            sims = np.dot(norm_rho_t, query_vec)
            
            top_k = np.argsort(sims)[-10:][::-1]
            print(f"\n--- Century {actual_centuries[t_idx]} ---")
            for idx in top_k:
                print(f"  {labels[idx]:<20} {sims[idx]:.4f}")


Loading data from fits/dbe_sinhala_dynamic/EF_EMB_26_04_27_18_20_28/variational0.dat...

Nearest Neighbors Evolution: බුදුන්

--- Century 13 ---
  බුදුන්               1.0000
  විනිශ්ච              0.3864
  දෙදෙන                0.3649
  රාජ්                 0.3530
  උකාර                 0.3479
  බුද්ධ                0.3464
  සර                   0.3435
  පළ                   0.3393
  පමණ                  0.3338
  දැ                   0.3331

--- Century 14 ---
  බුදුන්               1.0000
  සියබස                0.4589
  ගත                   0.4486
  සර                   0.4176
  මිස                  0.4150
  ලිය                  0.4074
  අසල                  0.3732
  මු                   0.3702
  අවදි                 0.3630
  වදාර                 0.3605

--- Century 15 ---
  බුදුන්               1.0000
  නාම                  0.5287
  පමණ                  0.4825
  ලො                   0.4766
  අර්ථ                 0.4731
  පළ                   0.4723
  බුද්ධ                0.4703
  ගේ 